# Cleaning Açaí Data for Pará

This notebook cleans the provided IBGE CSV so it can be used for a GIS join. The goal is to keep only municipalities in Pará and retain the municipality code and açaí data needed for mapping.

## 1. Read and Inspect the CSV

First, I imported Python's built-in `csv` module and opened the original dataset. I printed the first several rows to inspect the structure of the file and identify the rows and columns needed for the assignment.

In [52]:
import csv

filename = "tabela1613.csv"

with open(filename, "r", encoding="utf-8-sig") as file:
    reader = csv.reader(file)
    rows = list(reader)

for row in rows[:10]:
    print(row)

['Tabela 1613 - Área destinada à colheita, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras permanentes']
['Variável - Rendimento médio da produção']
['Nível', 'Cód.', 'Município', 'Ano x Produto das lavouras permanentes']
['Nível', 'Cód.', 'Município', '2024']
['Nível', 'Cód.', 'Município', 'Total', '', 'Açaí']
['MU', '1100015', "Alta Floresta D'Oeste (RO)", '..', 'Quilogramas por Hectare', '6000', 'Quilogramas por Hectare']
['MU', '1100023', 'Ariquemes (RO)', '..', 'Quilogramas por Hectare', '9167', 'Quilogramas por Hectare']
['MU', '1100031', 'Cabixi (RO)', '..', 'Quilogramas por Hectare', '-', 'Quilogramas por Hectare']
['MU', '1100049', 'Cacoal (RO)', '..', 'Quilogramas por Hectare', '-', 'Quilogramas por Hectare']
['MU', '1100056', 'Cerejeiras (RO)', '..', 'Quilogramas por Hectare', '-', 'Quilogramas por Hectare']


## 2. Filter to Municipalities in Pará

The original CSV contains municipalities from across Brazil as well as extra header and footer information. I kept only rows identified as municipality-level records (`MU`) and then filtered the municipality names to keep only those ending in `(PA)`, which identifies the state of Pará.

In [45]:
municipality_rows = [row for row in rows
                      if len(row) >= 7 and row[0] == "MU"]

para_rows = [row for row in municipality_rows
             if row[2].endswith("(PA)")]

print("Municipality rows:", len(municipality_rows))
print("Pará rows:", len(para_rows))

for row in para_rows[:10]:
    print(row)

Municipality rows: 5541
Pará rows: 143
['MU', '1500107', 'Abaetetuba (PA)', '..', 'Quilogramas por Hectare', '5600', 'Quilogramas por Hectare']
['MU', '1500131', 'Abel Figueiredo (PA)', '..', 'Quilogramas por Hectare', '8000', 'Quilogramas por Hectare']
['MU', '1500206', 'Acará (PA)', '..', 'Quilogramas por Hectare', '12000', 'Quilogramas por Hectare']
['MU', '1500305', 'Afuá (PA)', '..', 'Quilogramas por Hectare', '-', 'Quilogramas por Hectare']
['MU', '1500347', 'Água Azul do Norte (PA)', '..', 'Quilogramas por Hectare', '10000', 'Quilogramas por Hectare']
['MU', '1500404', 'Alenquer (PA)', '..', 'Quilogramas por Hectare', '4961', 'Quilogramas por Hectare']
['MU', '1500503', 'Almeirim (PA)', '..', 'Quilogramas por Hectare', '8000', 'Quilogramas por Hectare']
['MU', '1500602', 'Altamira (PA)', '..', 'Quilogramas por Hectare', '8000', 'Quilogramas por Hectare']
['MU', '1500701', 'Anajás (PA)', '..', 'Quilogramas por Hectare', '10000', 'Quilogramas por Hectare']
['MU', '1500800', 'Anani

## 3. Select the Required Columns

For the GIS join, only the municipality code and the açaí value are needed. I selected these two fields from each Pará municipality and removed the unnecessary columns.

In [46]:
selected_data = []

for row in para_rows:
    codigo = row[1]
    acai = row[5]
    selected_data.append([codigo, acai])

for row in selected_data[:10]:
    print(row)

['1500107', '5600']
['1500131', '8000']
['1500206', '12000']
['1500305', '-']
['1500347', '10000']
['1500404', '4961']
['1500503', '8000']
['1500602', '8000']
['1500701', '10000']
['1500800', '4185']


## 4. Check for Data Problems

I checked the açaí column for values that were not numeric. This identified symbols such as `-` and `...`, which need to be interpreted using the legend included with the IBGE dataset.

In [47]:
non_numeric = []

for row in selected_data:
    codigo = row[0]
    acai = row[1]

    if not acai.isnumeric():
        non_numeric.append(row)

print("Non-numeric rows:", len(non_numeric))

for row in non_numeric:
    print(row)

Non-numeric rows: 25
['1500305', '-']
['1501253', '-']
['1502509', '...']
['1502707', '-']
['1503002', '-']
['1503044', '-']
['1504422', '...']
['1505304', '-']
['1505551', '-']
['1505635', '-']
['1505650', '-']
['1506104', '-']
['1506138', '-']
['1506161', '...']
['1506195', '-']
['1506203', '-']
['1506559', '-']
['1506708', '...']
['1506906', '-']
['1507300', '-']
['1507755', '-']
['1507805', '-']
['1507904', '-']
['1507979', '-']
['1508407', '-']


## 5. Clean the Açaí Values

According to the IBGE legend, `-` represents an absolute zero, so these values were converted to `0`. The symbol `...` represents unavailable data, so these values were treated as missing. All remaining açaí values were converted from text to integers.

In [48]:
cleaned_data = []

for row in selected_data:
    codigo = row[0]
    acai = row[1]

    if acai == "-":
        acai = 0

    elif acai == "...":
        acai = None

    else:
        acai = int(acai)

    cleaned_data.append([codigo, acai])

for row in cleaned_data[:10]:
    print(row)

['1500107', 5600]
['1500131', 8000]
['1500206', 12000]
['1500305', 0]
['1500347', 10000]
['1500404', 4961]
['1500503', 8000]
['1500602', 8000]
['1500701', 10000]
['1500800', 4185]


In [49]:
missing_count = 0
non_numeric_count = 0

for row in cleaned_data:
    acai = row[1]

    if acai is None:
        missing_count += 1

    elif not isinstance(acai, int):
        non_numeric_count += 1

print("Total rows:", len(cleaned_data))
print("Missing values:", missing_count)
print("Non-numeric values remaining:", non_numeric_count)

Total rows: 143
Missing values: 4
Non-numeric values remaining: 0


## 6. My Cleaning Decisions

I filtered the original dataset to keep only municipality-level records in Pará (PA). I then kept only the municipality code and the açaí data column. Non-numeric values were checked using the IBGE legend. Values represented by - were changed to 0 because the legend defines this symbol as an absolute zero. Values represented by ... were treated as missing because the legend defines them as unavailable. The municipality codes were kept as text because they are identifiers that will be used for a GIS join, while the açaí values were converted to integers.

In [50]:
output_file = "para_acai_cleaned.csv"

with open(output_file, "w", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)

    writer.writerow(["municipality_code", "acai_production"])

    for row in cleaned_data:
        writer.writerow(row)

print("Cleaned CSV saved as:", output_file)

Cleaned CSV saved as: para_acai_cleaned.csv


## AI Usage Statement

I used ChatGPT to help me organize my approach to the assignment, troubleshoot errors while reading the CSV file, and develop parts of the filtering, cleaning, and export code. It also helped me understand how specific lines of code worked so I could revise and explain them in my notebook.

- reading and navigating the CSV structure
- identifying and cleaning non-numeric values
- troubleshooting file paths and Python errors